<a href="https://colab.research.google.com/github/erenozelll/Earthquake_Prediction/blob/main/Earthquake_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Libraries and Environment Setup
import pandas as pd
import numpy as np
import os

# Ensure the required directory structure exists for the workflow
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print("Kütüphaneler yüklendi ve klasörler hazır.")

Kütüphaneler yüklendi ve klasörler hazır.


In [ ]:
#Data Ingestion and Column Standardisation
file_path = '/content/data/raw/koeri_raw_v2.txt'

# Attempt to load the tab-separated file handling potential encoding variations
try:
    df = pd.read_csv(file_path, sep='\t', encoding='windows-1254')
except Exception:
    df = pd.read_csv(file_path, sep='\t', encoding='iso-8859-9')

# Trim whitespace from column headers to prevent indexing errors
df.columns = df.columns.str.strip()

# Subset feature columns relevant to the pipeline and standardise naming convention
df = df[['Olus tarihi', 'Olus zamani', 'Enlem', 'Boylam', 'Der(km)', 'MD', 'ML', 'Mw', 'Yer']]
df.columns = ['Olus_tarihi', 'Olus_zamani', 'Enlem', 'Boylam', 'Der_km', 'MD', 'ML', 'Mw', 'Yer']

print("Sekme (TAB) ayracı ile veri mükemmel şekilde okundu!")
display(df.head())
display(df.tail())

Sekme (TAB) ayracı ile veri mükemmel şekilde okundu!


,Olus_tarihi,Olus_zamani,Enlem,Boylam,Der_km,MD,ML,Mw,Yer
0,2024.12.31,10:46:27.00,38.0710,37.5068,6.7,0.0,4.0,3.9,TATLAR-NURHAK (KAHRAMANMARAS) [North West 8.1...
1,2024.12.30,20:30:27.56,39.4170,37.2977,5.0,0.0,4.3,4.3,KERTMEKARACAOREN-ULAS (SIVAS) [South West 2.1...
2,2024.12.30,02:25:17.55,36.7442,30.1467,7.1,0.0,3.9,3.8,OVACIK-ELMALI (ANTALYA) [South West 6.4 km]
3,2024.12.29,00:15:10.00,35.1408,27.2722,10.6,0.0,4.1,4.3,AKDENIZ
4,2024.12.28,11:44:00.95,41.2442,44.0308,5.0,0.0,3.6,3.7,GURCISTAN


,Olus_tarihi,Olus_zamani,Enlem,Boylam,Der_km,MD,ML,Mw,Yer
22787,1901.04.01,00:00:01.00,38.4,31.4,5.0,5.0,0.0,NaN,ATAKENT-AKSEHIR (KONYA) [North East 2.4 km]
22788,1901.03.01,00:00:01.00,38.2,27.7,5.0,5.0,0.0,NaN,YAKACIK-BAYINDIR (IZMIR) [South West 0.8 km]
22789,1901.02.23,00:00:00.00,37.9,27.9,15.0,4.7,4.6,4.8,KENGER- (AYDIN) [North East 1.1 km]
22790,1900.09.20,00:00:01.00,37.8,29.1,5.0,5.0,0.0,NaN,DENIZLI (DENIZLI) [North East 2.3 km]
22791,1900.04.17,02:23:00.00,42.2,45.1,35.0,4.7,4.6,4.8,GÜRCISTAN


In [ ]:

#Data Type Conversion and Magnitude Hierarchy Construction

#Combine date and time strings into a unified single datetime object
df['datetime'] = pd.to_datetime(df['Olus_tarihi'] + ' ' + df['Olus_zamani'], errors='coerce')

#Cast geospatial and physical features into numeric (float) types
numeric_cols = ['Enlem', 'Boylam', 'Der_km', 'MD', 'ML', 'Mw']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

#Implement Magnitude Priority Logic (Mw > ML > MD) based on seismological reliability
conditions = [
    df['Mw'].notna() & (df['Mw'] > 0),
    df['ML'].notna() & (df['ML'] > 0),
    df['MD'].notna() & (df['MD'] > 0)
]
choices = [df['Mw'], df['ML'], df['MD']]
df['magnitude'] = np.select(conditions, choices, default=np.nan)

#Standardise spatial feature names into English to match ML pipeline standards
df = df.rename(columns={
    'Enlem': 'lat',
    'Boylam': 'lon',
    'Der_km': 'depth_km'
})

print("Tip dönüşümleri ve magnitüd hiyerarşisi tamamlandı.")
display(df[['datetime', 'lat', 'lon', 'depth_km', 'magnitude']].head())

Tip dönüşümleri ve magnitüd hiyerarşisi tamamlandı.


,datetime,lat,lon,depth_km,magnitude
0,2024-12-31 10:46:27.000,38.0710,37.5068,6.7,3.9
1,2024-12-30 20:30:27.560,39.4170,37.2977,5.0,4.3
2,2024-12-30 02:25:17.550,36.7442,30.1467,7.1,3.8
3,2024-12-29 00:15:10.000,35.1408,27.2722,10.6,4.3
4,2024-12-28 11:44:00.950,41.2442,44.0308,5.0,3.7


In [ ]:
# Data Quality Assurance, Geospatial Filtering, and Serialization

# Record the initial dataset shape before executing filters
baslangic_boyutu = len(df)

# Feature selection: Extract only the target variables for the predictive models
df_clean = df[['datetime', 'lat', 'lon', 'depth_km', 'magnitude']].copy()

# Handle missing data by dropping rows with NaN values across selected features
df_clean = df_clean.dropna()

# Anomaly detection: Filter out physically implausible seismic records
df_clean = df_clean[
    (df_clean['magnitude'] > 0) & (df_clean['magnitude'] <= 9.0) &
    (df_clean['depth_km'] >= 0) & (df_clean['depth_km'] <= 700)
]

# Regional Bounding Box Filter: Restrict data to Turkey's seismic coordinates (35.0-43.0°N, 25.0-46.0°E)
df_clean = df_clean[
    (df_clean['lat'] >= 35.0) & (df_clean['lat'] <= 43.0) &
    (df_clean['lon'] >= 25.0) & (df_clean['lon'] <= 46.0)
]

# Deduplication: Remove redundant entries sharing identical timestamps and spatial coordinates
df_clean = df_clean.drop_duplicates(subset=['datetime', 'lat', 'lon'])

# Time-Series Alignment: Sort data chronologically and reset the global dataframe index
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)

bitis_boyutu = len(df_clean)

print("--- VERİ TEMİZLİĞİ ÖZETİ ---")
print(f"Başlangıçtaki Satır Sayısı: {baslangic_boyutu}")
print(f"Kutu Dışı ve Hatalı Silinen: {baslangic_boyutu - bitis_boyutu}")
print(f"Eğitime Hazır Temiz Veri: {bitis_boyutu}")

# Serialize the processed dataset to CSV format for pipeline interoperability
output_path = 'data/processed/koeri_clean.csv'

# Export to storage while omitting the auto-generated pandas index column
df_clean.to_csv(output_path, index=False)

print(f"\n Aşama 1 Başarılı! Temiz veri '{output_path}' konumuna kaydedildi.")

--- VERİ TEMİZLİĞİ ÖZETİ ---
Başlangıçtaki Satır Sayısı: 22792
Kutu Dışı ve Hatalı Silinen: 2394
Eğitime Hazır Temiz Veri: 20398

 Aşama 1 Başarılı! Temiz veri 'data/processed/koeri_clean.csv' konumuna kaydedildi.


In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from the CSV file instead of using a Parquet file
df = pd.read_csv('data/processed/koeri_clean.csv')

# Convert the datetime column back to a proper datetime format after reading it from the CSV file
df['datetime'] = pd.to_datetime(df['datetime'])

# Ensure that the dataset is correctly sorted in chronological order
df = df.sort_values('datetime').reset_index(drop=True)

df['target'] = (df['magnitude'] >= 4.2).astype(int)

oran = df['target'].mean() * 100
print(f"Hedef Değişken (Target) Oluşturuldu.")
print(f"Toplam Deprem: {len(df)}")
print(f"4.2 Üzeri Deprem Sayısı (Pozitif Sınıf): {df['target'].sum()}")
print(f"Pozitif Sınıf Oranı: %{oran:.2f}\n")

df['time_since_last_event_hours'] = df['datetime'].diff().dt.total_seconds() / 3600.0
df['prev_magnitude'] = df['magnitude'].shift(1)

print("✅ Geçmiş (Lag) özellikleri eklendi.")
display(df[['datetime', 'magnitude', 'target', 'time_since_last_event_hours', 'prev_magnitude']].head())
display(df[['datetime', 'magnitude', 'target', 'time_since_last_event_hours', 'prev_magnitude']].tail())


Hedef Değişken (Target) Oluşturuldu.
Toplam Deprem: 20398
4.2 Üzeri Deprem Sayısı (Pozitif Sınıf): 5153
Pozitif Sınıf Oranı: %25.26

✅ Geçmiş (Lag) özellikleri eklendi.


,datetime,magnitude,target,time_since_last_event_hours,prev_magnitude
0,1900-04-17 02:23:00,4.8,1,NaN,NaN
1,1900-09-20 00:00:01,5.0,1,3741.616944,4.8
2,1901-02-23 00:00:00,4.8,1,3743.999722,5.0
3,1901-03-01 00:00:01,5.0,1,144.000278,4.8
4,1901-04-01 00:00:01,5.0,1,744.000000,5.0


,datetime,magnitude,target,time_since_last_event_hours,prev_magnitude
20393,2024-12-28 11:44:00.950,3.7,0,40.923881,3.5
20394,2024-12-29 00:15:10.000,4.3,1,12.519181,3.7
20395,2024-12-30 02:25:17.550,3.8,0,26.168764,4.3
20396,2024-12-30 20:30:27.560,4.3,1,18.086114,3.8
20397,2024-12-31 10:46:27.000,3.9,0,14.266511,4.3


In [13]:
import pandas as pd
import numpy as np

# Since Pandas rolling works based on the index, we set datetime as the index
df_roll = df.set_index('datetime').copy()

# Our rolling windows: 7 days and 30 days
windows = ['7D', '30D']

for w in windows:
    # 1. Total number of earthquakes within the selected time window
    # closed='left' is very important!
    # It means: "Do not count the current earthquake, only count the previous ones."
    count_col = f'count_events_{w}'
    df_roll[count_col] = df_roll['magnitude'].rolling(window=w, closed='left').count()

    # 2. Maximum magnitude within the selected time window
    max_col = f'max_mag_{w}'
    df_roll[max_col] = df_roll['magnitude'].rolling(window=w, closed='left').max()

    # 3. Average magnitude within the selected time window
    mean_col = f'mean_mag_{w}'
    df_roll[mean_col] = df_roll['magnitude'].rolling(window=w, closed='left').mean()

# Reset the index and bring datetime back as a column
df = df_roll.reset_index()

# Rolling calculations produce missing values (NaN) for the first rows
# because there is no previous data available for them.
# In this project, machine learning algorithms, especially the baseline Logistic Regression,
# do not handle NaN values well.
print(f"Number of rows before rolling: {len(df)}")

df_features = df.dropna().reset_index(drop=True)

print(f"Number of rows after removing missing feature values: {len(df_features)}")

print("\n✅ Rolling window features were added and data leakage was prevented!")

display(df_features[['datetime', 'magnitude', 'count_events_7D', 'max_mag_30D']].tail())

Number of rows before rolling: 20398
Number of rows after removing missing feature values: 19297

✅ Rolling window features were added and data leakage was prevented!


,datetime,magnitude,count_events_7D,max_mag_30D
19292,2024-12-28 11:44:00.950,3.7,6.0,4.7
19293,2024-12-29 00:15:10.000,4.3,7.0,4.7
19294,2024-12-30 02:25:17.550,3.8,7.0,4.7
19295,2024-12-30 20:30:27.560,4.3,8.0,4.7
19296,2024-12-31 10:46:27.000,3.9,9.0,4.7


In [14]:
import pandas as pd

# 1. Save the final feature matrix
output_features = 'data/processed/features.csv'
df_features.to_csv(output_features, index=False)
print(f"✅ Final özellik matrisi kaydedildi: {output_features}")

# 2. Remove columns that are not required for the model
# The datetime column will only be used for splitting the dataset and will not be included in the model.
# The magnitude column represents the target information itself, so including it in the model would cause serious data leakage.
drop_cols = ['datetime', 'magnitude', 'target']
X = df_features.drop(columns=drop_cols)
y = df_features['target']
tarihler = df_features['datetime']

# 3. Chronological split based on the project plan dates
# Train: 2000 - end of 2019
# Validation: 2020 - end of 2022
# Test: 2023 - end of 2024

train_mask = (tarihler >= '2000-01-01') & (tarihler <= '2019-12-31')
val_mask = (tarihler >= '2020-01-01') & (tarihler <= '2022-12-31')
test_mask = (tarihler >= '2023-01-01') & (tarihler <= '2024-12-31')

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("\n--- Veri Seti Bölünme Özeti ---")
print(f"Eğitim (Train) Seti: {len(X_train)} satır")
print(f"Doğrulama (Val) Seti: {len(X_val)} satır")
print(f"Test Seti: {len(X_test)} satır (Sona kadar dokunulmayacak!)")

✅ Final özellik matrisi kaydedildi: data/processed/features.csv

--- Veri Seti Bölünme Özeti ---
Eğitim (Train) Seti: 8404 satır
Doğrulama (Val) Seti: 1920 satır
Test Seti: 2360 satır (Sona kadar dokunulmayacak!)
